# Lesson 41 · The windowed-FFT pipeline — open gently, see clearly: run windowing + FFT end to end and watch the sidelobes drop

**Goal**: first understand why we window, then run the `windowed_fft` pipeline yourself and watch the sidelobes get pushed down.

> **Steps 1-2-3**: truncation leaks → multiply by Hann to push the sidelobes down → watch the spectrum clean up. It plugs into the windows from L36 and the spectrum reading from L40.

🔗 **Aurora connection**: `aurora.audio.stft()` windows every frame before taking its FFT; the window functions in `aurora/audio/windows.py` (`hann`, `hamming` and friends) are exactly the module this lesson wires in.

← **Previous lesson** · [L40 · Spectrum analysis in practice](L40_spectrum.ipynb)

> Last lesson covered **spectrum analysis in practice**: magnitude spectrum / phase spectrum / frequency resolution, with a 440 Hz + 880 Hz mix.
> This lesson explores **the windowed-FFT pipeline**.

## This lesson's story: why window? — fading the radio in sounds better than cutting it off

Imagine you're listening to a piece of music and it gets chopped off with a sudden "click". In frequency terms, that jump at the edge is like mixing in a burst of broadband noise. That's **spectral leakage**.

The FFT doesn't process the whole signal — it processes one frame. Cutting out a frame is the same as multiplying the signal by a rectangular window: it drops abruptly to zero at the edges, and energy smears across the spectrum. Multiplication in time → convolution in frequency: the sinc sidelobes of the rectangular window "leak" energy into neighboring frequency bins.

**The fix**: replace the rectangular window with a window that eases smoothly down to zero at the edges — the Hann window is the most common choice:

$$w[n] = 0.5\left(1 - \cos\!\frac{2\pi n}{N}\right), \quad n = 0,\ldots,N-1$$

The middle keeps the original signal and the two ends fade to zero, like slowly turning a radio up and back down again — no more abrupt jumps.

**The full pipeline** (three steps):

| Step | Operation | Effect |
|---|---|---|
| 1 | `frame = x[start:start+N]` | take a frame |
| 2 | `frame = frame * w` | multiply by the window |
| 3 | `X = fft(frame)` | FFT → complex spectrum |

Your task in this lesson: implement `windowed_fft(x, window_type="hann")` — one function that glues the three steps together.

### Going deeper: why does truncating in time cause convolution and leakage in frequency?

You might ask: why does truncation (a multiplication in time) turn into a convolution in frequency?

**The convolution theorem** says: if you multiply two signals in time, `a[n] × b[n]`, their Fourier transforms convolve in frequency: `A(ω) ∗ B(ω)`.

**The intuition here**:
- The spectrum `X(f)` of the original signal `x[n]` may be very "sharp" — a 440 Hz pure tone, for example, is a single spike on the frequency axis.
- Cutting out an FFT frame is the same as multiplying `x[n]` by a rectangular window: `x[n] × rect(n)`.
- The rectangular window has its own frequency-domain picture, the **sinc function** `sinc(f)`. A sinc isn't a spike: it's a "mountain" that's tall in the center and tapers off to both sides, with lots of small ripples (sidelobes) on either side.
- **What the convolution does**: convolving the original spike with this sinc mountain turns the spike into a broad hump, and together with the sinc's sidelobes, energy leaks into neighboring frequency bins.

**Analogy**: it's like convolving a thin line with a wide brush — you get a broad stroke, not a thin line. The rectangular window is that "wide brush".

**Mathematical properties (just note them for now; L43+ goes deeper)**:
- The Fourier transform of the rectangular window is a sinc function
- The sinc's main lobe (center) is about `4π/N` wide, and its sidelobe peaks sit about 13 dB below the main lobe (-13 dB)
- That's why the rectangular window leaks the most

**The fix**: replace the rectangular window with a window whose frequency-domain sidelobes are lower, such as the Hann window.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.windows import hann, hamming
from aurora.audio.transforms import fft as aurora_fft  # the Cooley-Tukey FFT you wrote by hand in L37-39

### About the imports: the FFT engine and the window functions

**When to use aurora_fft vs np.fft.fft?**
- **When implementing `windowed_fft`** (section 4): you must use `aurora_fft` (the Cooley-Tukey FFT you wrote by hand in L37-39), to show you understand it from scratch
- **For demo plots and parameter experiments** (throughout this lesson): for convenience and speed, `np.fft.rfft` or `np.fft.fft` are fine for plotting
- **When checking correctness**: use `np.fft.fft` as the reference, to make sure `aurora_fft` gives the same result

**What the window functions return**:
- `hann(N)` and `hamming(N)` both return a 1-D numpy array of shape `(N,)`
- the values lie in `[0, 1]`
- no manual normalization needed
- multiply them elementwise with a signal frame directly: `x_windowed = x_frame * hann(N)`

## 1. Spectral leakage

Truncated signal = original signal × rectangular window. Multiplication in time ↔ convolution in frequency, so the spectrum = original spectrum ∗ sinc. The sinc's sidelobes stretch out from the main lobe to both sides, spreading energy that should sit in one bin across all the bins. That's **spectral leakage**.

The rectangular window's sidelobe peak is only about 13 dB below the main lobe (`-13 dB`), which means the sidelobes of a strong signal are enough to drown out a weak signal next to it.

### What are the sinc function, the main lobe and the sidelobes?

The **sinc function** is defined as:
$$\text{sinc}(x) = \frac{\sin(\pi x)}{\pi x}$$

What does it look like?
- **In the middle (at x=0)**: a tall peak (value 1)
- **Decaying to both sides**: but not smoothly — it swings up and down
- **At x = ±1, ±2, ±3, ...**: it crosses zero
- **Main lobe**: the big "mountain" in the middle, from x = -1 to x = +1
- **Sidelobes**: the small ripples on either side, getting smaller and smaller but never disappearing

**The culprit behind spectral leakage**: the Fourier transform of the rectangular window is a sinc function. When you truncate a signal, those sidelobes sprinkle energy onto neighboring frequency bins, as if a sharp peak had been "blurred".

The code below shows the sinc for what it really is:

In [ ]:
# Visualize the sinc function
x_sinc = np.linspace(-8, 8, 2000)
# compute sinc; the 0/0 case at x=0 has to be handled
y_sinc = np.sinc(x_sinc)  # np.sinc already handles the singularity

plt.figure(figsize=(11, 4))
plt.plot(x_sinc, y_sinc, linewidth=2, label='sinc(x)')
plt.axhline(0, color=plt.rcParams["axes.edgecolor"], linewidth=0.5)
plt.axhline(-10**(-13/20), color=plt.rcParams["grid.color"], linestyle=':', linewidth=1, alpha=0.7, label='-13 dB line (rectangular-window sidelobe)')
plt.axvline(-1, color='orange', linestyle='--', linewidth=1, alpha=0.5)
plt.axvline(1, color='orange', linestyle='--', linewidth=1, alpha=0.5, label='main-lobe edge (x=±1)')

# mark the main lobe and the sidelobes
plt.fill_between(x_sinc[(x_sinc >= -1) & (x_sinc <= 1)], 0, y_sinc[(x_sinc >= -1) & (x_sinc <= 1)],
                  alpha=0.2, color='orange', label='main lobe')
plt.scatter([-2.46, -1.43, 1.43, 2.46], np.sinc(np.array([-2.46, -1.43, 1.43, 2.46])),
            color='red', s=30, zorder=5, label='sidelobe peaks')

plt.xlabel('x')
plt.ylabel('sinc(x)')
plt.title('The sinc function: the root cause of spectral leakage')
plt.legend(loc='upper right')
plt.ylim(-0.25, 1.1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("🔍 What to look for:")
print(f"  • Main lobe (-1 ≤ x ≤ 1): where most of the energy gathers")
print(f"  • First sidelobe (x ≈ ±1.43): about {abs(np.sinc(1.43)):.3f} of the main lobe, roughly -13 dB")
print(f"  • Sidelobes farther out keep shrinking but never vanish")
print(f"  • This sidelobe behavior of the rectangular window is the root of spectral leakage")

In [ ]:
# Demo: spectral leakage of a sine wave that doesn't land exactly on an integer bin
N = 256
fs = 1000.0
# 440 Hz is not an integer multiple of fs/N → the frequency doesn't line up with a bin
t = np.arange(N) / fs
x = np.sin(2 * np.pi * 440 * t)

X_rect = np.abs(np.fft.rfft(x))
freqs = np.fft.rfftfreq(N, 1/fs)

plt.figure(figsize=(10, 3))
plt.plot(freqs, 20 * np.log10(X_rect + 1e-12))
plt.axvline(440, color='r', linestyle='--', label='440 Hz')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Rectangular-window FFT — clearly visible sidelobe leakage')
plt.legend()
plt.tight_layout()
plt.show()

## From the rectangular window to the Hann window: the design philosophy of window functions

We already know what's wrong with the rectangular window: hard edges and high sidelobes in frequency (-13 dB). **So how do you design a better window?**

**Three goals of window design**:
1. **Smooth edges**: the window should be close to zero at both ends and 1 in the middle, with no steep jumps
2. **Low sidelobes in frequency**: the window's Fourier transform should have sidelobes as low as possible, to reduce leakage
3. **A main lobe that isn't too wide**: resolution and sidelobe suppression have to be balanced

**How the Hann window is designed**: in signal processing, people noticed a handy property of the **cosine wave** — its derivative is zero at both ends. So if you build a smoothly falling weight out of a cosine, you get "smooth edges" for free. The Hann window formula is built on exactly this idea:
$$w[n] = 0.5 \times (1 - \cos(...))$$

What's neat about this formula:
- when `cos(·) = 1` (at the ends), `w[n] = 0`
- when `cos(·) = -1` (in the middle), `w[n] = 1`
- the transition in between is very smooth, with no sharp corners

**Periodic vs symmetric**:

The Hann window comes in two versions:
- **Periodic** (the one this lesson uses): the denominator is `N`, as if the window repeated forever
  - `w[n] = 0.5(1 - cos(2πn/N))`, n = 0, 1, ..., N-1
  - the right end `w[N-1] ≠ 0`; it mirrors `w[1]`
  - **used in STFT/COLA**, because joining frame to frame needs periodicity

- **Symmetric** (used in some cases): the denominator is `N-1`, as if the window had a definite start and end, perfectly symmetric
  - `w[n] = 0.5(1 - cos(2π×n/(N-1)))`, n = 0, 1, ..., N-1
  - both ends are exactly zero
  - **used outside frame-by-frame analysis**, for example when processing a discrete signal in one piece (it's also the usual choice for FIR filter design)

This lesson uses the periodic version because the FFT will soon feed into the STFT (Short-Time Fourier Transform), where frames have to be joined seamlessly with OLA (overlap-add).

## 2. The Hann window

The Hann window is defined as (periodic version):

$$w[n] = 0.5 \times \left(1 - \cos\frac{2\pi n}{N}\right), \quad n = 0, 1, \ldots, N-1$$

**Working out the edge values exactly**:
- When `n=0`: `cos(0) = 1`, so `w[0] = 0.5 × (1 - 1) = 0` ✓
- When `n=N-1` (e.g. n=255 for N=256):
  - `cos(2π×255/256) = cos(2π - 2π/256)`
  - since `cos(2π - ε) = cos(ε) ≈ 1` (when ε is small)
  - we get `w[N-1] ≈ 0.5 × (1 - 0.9999...) ≈ 0.0001...` — very close to 0 but not exactly 0
  - that's the hallmark of the **periodic** window: the right end isn't exactly zero, just very close. The larger N is, the closer w[N-1] gets to 0

**Performance compared with the rectangular window**:

| Metric | Rectangular | Hann |
|---|---|---|
| Main-lobe width | `4π/N` | `8π/N` (2× wider) |
| First sidelobe level | -13 dB | -31 dB |
| Sidelobe suppression | poor | excellent (18 dB lower than rectangular) |

**Where the dB values come from**: these numbers are the windows' **theoretical frequency-domain properties**. The DTFT (discrete-time Fourier transform) of the rectangular window is exactly a sinc function, whose first sidelobe is about -13.3 dB in theory; working out the Hann window's frequency response puts its first sidelobe at about -31.5 dB. These are **fixed mathematical properties** of the windows, not measurements.

`hann(N)` in `aurora/audio/windows.py` returns a numpy array of length N with values in [0, 1]; no extra normalization needed.

In [ ]:
# Check the edge values of the Hann window
N_test = [8, 64, 256]
for N in N_test:
    n = np.arange(N)
    w = 0.5 * (1 - np.cos(2 * np.pi * n / N))
    print(f"N={N:3d}: w[0]={w[0]:.6f}, w[N-1]={w[-1]:.6f}, w[1]={w[1]:.6f}")
    # look at the periodicity: w[N-1] should match w[1] (mirror images)
    if N > 2:
        print(f"         w[N-1] mirrors w[1]? difference={abs(w[-1] - w[1]):.2e}")

print("\n📌 Takeaway: the periodic Hann window's right end w[N-1] ≠ 0, but it shrinks toward 0 as N grows")

In [ ]:
# Compare the time-domain shapes of three windows
N = 64
n = np.arange(N)
w_rect = np.ones(N)
w_hann = hann(N)
w_hamm = hamming(N)

plt.figure(figsize=(10, 3))
plt.plot(n, w_rect, label='Rectangular')
plt.plot(n, w_hann, label='Hann')
plt.plot(n, w_hamm, label='Hamming', linestyle='--')
plt.xlabel('Sample n')
plt.ylabel('Weight')
plt.title('Time-domain shapes of three windows (N=64)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. The windowed-FFT pipeline

The standard pipeline has three steps:

```
1. frame     = x_segment          # take one frame from the signal (length N)
2. x_win     = frame * window      # multiply elementwise by the window (same length)
3. spectrum  = FFT(x_win)          # FFT the windowed frame; take the magnitude for the magnitude spectrum
```

In the STFT these three steps repeat for every frame, with consecutive frames offset by hop_length. That loop is exactly what's inside `aurora.audio.stft()`.

In [ ]:
# Show how windowing smooths the frame edges
N = 128
t = np.arange(N) / 1000.0
x_seg = np.sin(2 * np.pi * 440 * t)   # a frame that doesn't line up
w = hann(N)
x_win = x_seg * w

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(x_seg, label='original frame')
axes[0].set_title('Truncated frame (jumps at the edges)')
axes[0].set_xlabel('Sample')
axes[1].plot(x_win, label='after the Hann window', color='orange')
axes[1].set_title('After the Hann window (smooth edges)')
axes[1].set_xlabel('Sample')
plt.tight_layout()
plt.show()

## 4. ✏️ Implement `windowed_fft(x, window_type="hann")`

**The core task**: pick the window based on `window_type`, multiply the input frame by it, then take the FFT:

| Step | Code idea | Notes |
|---|---|---|
| 1 | `if window_type == "hann": w = hann(N)` | imported from `aurora.audio.windows` |
|   | `elif window_type == "hamming": w = hamming(N)` | |
|   | `elif window_type == "rectangular": w = np.ones(N)` | leaves the signal unchanged; any other type → `raise ValueError` |
| 2 | `return aurora_fft(x * w)` | iron rule: the engine is the Cooley-Tukey FFT you wrote by hand in L37-39 (every N in this lesson is a power of 2); `np.fft.fft` is only for cross-checking in the check cells |

**Acceptance criteria**:
- `window_type="rectangular"` → cross-checks equal to `np.fft.fft(x)` (because `np.ones(N) * x = x`; np.fft is only the reference)
- `window_type="hann"` → output is a complex array of shape `(N,)`
- `window_type` you don't recognize → `ValueError`
- Sidelobe level: in the section 5 experiment, Hann's sidelobes come out roughly 30 dB below the rectangular window's

In [ ]:
def windowed_fft(x: np.ndarray, window_type: str = "hann") -> np.ndarray:
    """Apply the chosen window to the input frame x, then take the FFT.

    Parameters
    ----------
    x           : input frame, shape (N,)
    window_type : window type, "hann" | "hamming" | "rectangular"

    Returns
    -------
    complex FFT output, shape (N,)
    """
    N = len(x)
    # ✏️ TODO: build the window vector from window_type (hann / hamming / ones)
    # Hint: an unknown type should raise ValueError(f"unknown window type: {window_type}")
    # Hint: the FFT engine is aurora_fft, imported at the top (from scratch; works when N is a power of 2);
    #      np.fft.fft is only for cross-checking in the check cells, not the engine.
    raise NotImplementedError("Implement windowed_fft")


In [ ]:
# Check: a rectangular window doesn't change x, so the result equals a plain FFT
try:
    _out = windowed_fft(np.ones(64), 'rectangular')
    assert np.allclose(_out, np.fft.fft(np.ones(64))), 'rectangular-window FFT should equal np.fft.fft'
    print('✅ windowed_fft(rectangular) passed')

    x_test = np.random.randn(128)
    out = windowed_fft(x_test, 'hann')
    assert out.shape == (128,), f'wrong output shape: {out.shape}'
    print('✅ windowed_fft(hann) output shape is correct')

    w = hann(128)
    assert np.isclose(out[0].real, np.sum(x_test * w)), 'bin 0 should equal the sum of the windowed signal'
    print('✅ bin 0 = sum(x * window) check passed')

    # Major fix: add a test for the hamming path
    out_h = windowed_fft(x_test, 'hamming')
    assert out_h.shape == (128,), f'wrong hamming output shape: {out_h.shape}'
    print('✅ windowed_fft(hamming) output shape is correct')

    # Major fix: an unknown window_type should raise ValueError
    try:
        windowed_fft(x_test, 'invalid_window')
        print('❌ an unknown window_type should raise ValueError, but nothing was raised')
    except ValueError as e:
        print(f'✅ unknown window_type correctly raises ValueError: {e}')
    except (NotImplementedError, TypeError):
        print('⬜ Finish windowed_fft completely (including the ValueError branch), then run this cell')

except NotImplementedError:
    print('⬜ Implement windowed_fft first, then run this cell')
except TypeError:
    print('⬜ windowed_fft returned something that is not an array; finish the TODO')


## 5. Parameter experiment: comparing sidelobes, and frequency alignment

### The arithmetic of frequency alignment

The FFT maps a time-domain signal onto discrete frequency bins. The **frequency resolution** is defined as:
$$\Delta f = \frac{f_s}{N}$$
where $f_s$ is the sample rate and $N$ is the number of FFT points.

**What do "aligned" and "misaligned" mean?**
- If the frequency $f$ you want to analyze is exactly an integer multiple of $\Delta f$, it's **aligned**, and the energy sits in a single bin
- If $f$ is not an integer multiple, it's **misaligned**, and the energy leaks into several neighboring bins

**A concrete example** (the settings of this section's parameter experiment):
- sample rate $f_s = 1000$ Hz, FFT size $N = 256$
- frequency resolution $\Delta f = 1000 / 256 \approx 3.906$ Hz
- target frequency 440 Hz: $440 / 3.906 \approx 112.6$ (**not an integer**), so it leaks badly
- to choose an aligned frequency we need $f = k \times \Delta f$ with $k$ an integer. For example $k = 112$:
  $$f = 112 \times 3.90625 = 437.5 \text{ Hz}$$

**Another way to see it**: within the sampling time $T = N/f_s = 256/1000 = 0.256$ seconds, the signal has to complete a whole number of cycles.
- For 440 Hz: number of cycles = $440 \times 0.256 = 112.64$ (not an integer → leakage)
- For 437.5 Hz: number of cycles = $437.5 \times 0.256 = 112$ (an integer → aligned)

### Warning: the Nyquist frequency

The **Nyquist frequency** $f_N = f_s / 2$ is the limiting frequency of the sampling theorem. A sine wave at exactly this frequency (with zero phase) **vanishes completely** once sampled.

**Why?** Suppose you sample a sine wave whose frequency is $f_N$:
$$x[n] = \sin(2\pi f_N \cdot n/f_s) = \sin(2\pi \cdot (f_s/2) \cdot n/f_s) = \sin(\pi n)$$
and $\sin(\pi n) = 0$ for every integer $n$, so every sample is zero!

**Heads-up for this section**: in this parameter experiment $f_s = 1000$ Hz, so $f_N = 500$ Hz. Whatever you do, don't pick 500 Hz to test with, or you won't see any spectrum at all.

**Summary of what you'll see**:
- **Misaligned** (e.g. 440 Hz): the rectangular window's sidelobes are high (-13 dB), the Hann window's are low (-31 dB), and the difference is visible to the naked eye
- **Aligned** (e.g. 437.5 Hz): the leakage disappears; the rectangular window collapses to a single spectral line and the Hann window shrinks to 3 neighboring lines (its main lobe is twice as wide)
- Why do the **sidelobe values in the experiment ("excluding ±5 bins around the main lobe")** differ from **the -13/-31 dB quoted at the start**?
  - The -13 dB and -31 dB at the start are **theoretical values** for each window's **first sidelobe** (right next to the main lobe)
  - The -24 dB and -55 dB measured in the experiment after excluding ±5 bins are **sidelobes farther out**, which have already decayed
  - Both sets of numbers are right; they're just measured in different places

In [ ]:
# Parameter experiment: rectangular vs hann sidelobes
freq_hz = 440.0   # change this to see aligned vs misaligned (try 437.5: exactly 112 whole cycles)
fs = 1000.0
N = 256

t = np.arange(N) / fs
x = np.sin(2 * np.pi * freq_hz * t)

try:
    X_rect = np.abs(windowed_fft(x, window_type="rectangular")[:N//2])
    X_hann = np.abs(windowed_fft(x, window_type="hann")[:N//2])
except (NotImplementedError, TypeError):
    print('⬜ Finish windowed_fft in section 4 first, then run this experiment')
else:
    freqs = np.fft.rfftfreq(N, 1/fs)[:-1]   # N//2 points

    # normalize: put the main-lobe peak at 0 dB
    X_rect_db = 20 * np.log10(X_rect / X_rect.max() + 1e-12)
    X_hann_db = 20 * np.log10(X_hann / X_hann.max() + 1e-12)

    # sidelobe difference (excluding ±5 bins around the main lobe)
    peak_bin = np.argmax(X_rect)
    mask = np.ones(len(X_rect), dtype=bool)
    mask[max(0, peak_bin-5):peak_bin+6] = False
    sidelobe_rect = X_rect_db[mask].max()
    sidelobe_hann = X_hann_db[mask].max()
    print(f'Rectangular max sidelobe: {sidelobe_rect:.1f} dB')
    print(f'Hann max sidelobe:        {sidelobe_hann:.1f} dB')
    print(f'Sidelobe suppression: {sidelobe_rect - sidelobe_hann:.1f} dB')

    plt.figure(figsize=(11, 4))
    plt.plot(freqs, X_rect_db, label='Rectangular', alpha=0.8)
    plt.plot(freqs, X_hann_db, label='Hann', alpha=0.8)
    plt.axhline(-13, color=plt.rcParams["grid.color"], linestyle=':', linewidth=0.8, label='-13 dB reference')
    plt.axhline(-31, color=plt.rcParams["grid.color"], linestyle=':', linewidth=0.8, label='-31 dB reference')
    plt.ylim(-80, 5)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Normalized magnitude (dB)')
    plt.title(f'Spectral leakage compared ({freq_hz} Hz, fs={fs} Hz, N={N})')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Wrap-up

In this lesson you implemented `windowed_fft(x, window_type)`, which windows a frame and returns its length-N complex FFT spectrum. The Hann window pushes the sidelobes down by about 18 dB (a sidelobe peak of about −31 dB, against about −13 dB for the rectangular window), all thanks to the set of cosine weights in `aurora/audio/windows.py`. The next lesson (L42 · The FFT drawn) draws this computation side by side with the spectra of a pure tone, a chord and noise.

## ✏️ Whiteboard challenge: windowed FFT by hand and by reasoning (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: What is the root cause of spectral leakage?
(Hint: truncating in time = multiplying by a rectangular window → what operation happens in frequency?)

**Question 2**: What are the edge values of the Hann window `w[n] = 0.5(1 - cos(2πn/N))`?
(w[0]=? at n=0, w[N-1]=? at n=N-1)

**Question 3**: What does `windowed_fft([1, 1, 1, ..., 1], window_type="rectangular")` return?
(An all-ones signal has only a DC component: `X[0]=N`, and every other `X[k]=0`)

**Question 4**: The Hann window's first sidelobe sits about 31 dB below its main lobe. How many times smaller is that in amplitude?
(Hint: 20·log₁₀(ratio) = -31 dB → ratio = 10^(-31/20) ≈ ?)

**Question 5**: Why do adjacent frames in the STFT (short-time Fourier transform) usually overlap by 50%?
(Hint: the Hann window is largest in the middle and close to zero at both ends — where does the window suppress information?)

When you've worked it out, run the cell below to check your answers.

### Going deeper on Question 5: the OLA (overlap-add) condition

Question 5 asks "why use 50% overlap in the STFT?". The answer involves one of the core tricks of signal processing: the **COLA condition** (Constant Overlap-Add).

**Background**: in real-time audio processing (playback, noise reduction and so on), we can't process a whole audio file in one go; we have to work frame by frame. Each frame is analyzed with an FFT, processed, and turned back into the time domain with an IFFT (inverse Fourier transform). Finally the outputs of all the frames are stitched together — this is called **overlap-add**.

**The problem**: if you just stack frames end to end, there are "seams" at the boundaries and the sound becomes discontinuous. Because the Hann window is zero at both ends, the information at the edges of each frame gets suppressed. With only one copy of that information, it's lost.

**The fix**: let adjacent frames overlap by 50% (overlap = N/2). Then:
- frame 1: covers samples 0 to N-1
- frame 2: covers samples N/2 to 3N/2 − 1
- frame 3: covers samples N to 2N-1
- ...

**The COLA condition**: for the periodic Hann window with 50% overlap,
$$w[n] + w[n + N/2] = \text{constant} \quad \text{for all } n = 0, 1, \ldots, N/2-1$$

This means that whenever a sample is weighted by the windows of two adjacent frames and the results are added, the total weight is always the same constant. Call that constant C; then:
$$\text{reconstructed signal}[n] = C \times x[n]$$
No information is lost!

**Numerical check** (Q5 in the code): compute `w[:N/2] + w[N/2:]` and you'll see every entry equals the same constant (for the periodic Hann window, exactly 1), with a standard deviation of essentially zero.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: cause of leakage — multiply by a rectangular window in time ↔ convolve with sinc in frequency
# check: the DTFT of the rectangular window is a sinc
N = 16
rect = np.ones(N)
X_rect = np.fft.fft(rect, n=256)  # zero-pad to zoom in on the frequency domain
# the sinc sidelobes exist
assert np.max(np.abs(X_rect[8:128])) > 0.1  # sidelobes are nonzero
print(f"Q1 ✅  truncating in time = multiplying by a rectangular window; in frequency = convolving with a sinc, whose sidelobes spread energy into neighboring bins")

# Q2: Hann window edge values (periodic, denominator N)
N_h = 8
n_arr = np.arange(N_h)
w_hann = 0.5 * (1 - np.cos(2 * np.pi * n_arr / N_h))
assert np.isclose(w_hann[0], 0.0, atol=1e-12)
assert np.isclose(w_hann[-1], 0.5 * (1 - np.cos(2 * np.pi * (N_h - 1) / N_h)))
w0 = w_hann[0]
wNm1 = w_hann[-1]
print(f"Q2 ✅  w[0]={w0:.6f} (=0), w[N-1]={wNm1:.6f} (periodic window: right end ≠ 0, equals w[1]; closer to 0 as N grows)")

# Q3: rectangular window on an all-ones signal → DC component = N
N3 = 16
x_ones = np.ones(N3)
X3 = np.fft.fft(x_ones)
assert np.isclose(X3[0], N3 + 0j, atol=1e-12)
assert np.allclose(X3[1:], 0+0j, atol=1e-12)
try:
    X3_mine = windowed_fft(x_ones, "rectangular")
    assert np.allclose(X3_mine, X3, atol=1e-9), f"rectangular should match np.fft.fft"
    print(f"Q3 ✅  windowed_fft([1..1], 'rectangular')[0]={X3_mine[0]:.1f}, the rest ≈0")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  FFT of an all-ones signal: X[0]={X3[0]:.1f}=N, X[k≠0]=0 (pure DC) (windowed_fft not implemented yet)")

# Q4: -31 dB → amplitude ratio
db_diff = -31.0
ratio = 10 ** (db_diff / 20)
assert abs(ratio - 0.0282) < 0.01  # ≈ 0.028
print(f"Q4 ✅  -31 dB → amplitude ratio = 10^(-31/20) ≈ {ratio:.4f} (about 1/35 of the main-lobe peak)")

# Q5: why 50% overlap — it makes up for the information the Hann window suppresses at the ends
# check that with 50% overlap the Hann window's reconstruction weight is constant
N5 = 8
w5 = 0.5 * (1 - np.cos(2 * np.pi * np.arange(N5) / N5))
hop = N5 // 2
# OLA (overlap-add) reconstruction: w[n] + w[n + hop] for n=0..hop-1
ola_sum = w5[:hop] + w5[hop:]
# this should be constant (the Hann window with 50% overlap satisfies COLA)
assert np.std(ola_sum) < 0.05 * np.mean(ola_sum), f"OLA at 50% overlap is not flat: {ola_sum}"
print(f"Q5 ✅  with 50% overlap the Hann OLA sum ≈ {np.mean(ola_sum):.4f} (flat), so nothing at the ends is lost")
print("\n🎉 Windowed-FFT whiteboard challenge passed! windowed_fft = frame × window → fft.")

In [ ]:
# ✏️ Self-review for this lesson
l41_review = {
    "spectral_leakage_cause":  None,  # remember: leakage = rectangular truncation in time → sinc convolution in frequency? True/False
    "hann_window_formula":     None,  # remember w[n]=0.5(1-cos(2πn/N)), edges = 0? True/False
    "windowed_fft_impl":       None,  # windowed_fft implemented and passing the rectangular assertion? True/False
    "sidelobe_reduction":      None,  # understand that Hann's first sidelobe sits about 31 dB below the main lobe? True/False
    "whiteboard_passed":       None,  # whiteboard challenge worked on paper? True/False
}

unfilled = [k for k, v in l41_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l41_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L41 fully cleared! On to L42: the FFT drawn (visual lesson)')

---

→ **Next lesson** · [L42 · The FFT drawn](L42_visual_fft.ipynb)

> Next lesson covers **the FFT drawn**: the butterfly diagram, plus the spectra of a pure tone, a chord and noise side by side.